# Phase 2 — Exploratory Data Analysis

**Business question:** NexaTel is losing roughly a quarter of its subscriber base.
Who is leaving, and what do they have in common?

Everything here reads from `db/nexatel.db` through the SQL layer built in Phase 1 —
not from the raw CSV. In a real company the CSV does not exist; the database does.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np
pd.set_option('display.width', 120)

from config import DB_PATH
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query('SELECT * FROM v_customer_360', conn)
print(df.shape)
df.head(3)

(7043, 22)


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,churn_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1


## 1. Data quality — what is wrong with the extract before we trust it

In [2]:
raw_total = pd.to_numeric(df['total_charges'], errors='coerce')
print('rows                :', len(df))
print('duplicate ids       :', df.customer_id.duplicated().sum())
print('missing total_charges:', raw_total.isna().sum())
print('...all tenure = 0   :', (df.loc[raw_total.isna(), 'tenure'] == 0).all())
print('monthly_charges <= 0:', (df.monthly_charges <= 0).sum())
df.dtypes.to_frame('dtype').T

rows                : 7043
duplicate ids       : 0
missing total_charges: 11
...all tenure = 0   : True
monthly_charges <= 0: 0


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,churn_flag
dtype,str,str,int64,str,str,int64,str,str,str,str,...,str,str,str,str,str,str,float64,float64,str,int64


`total_charges` ships as text with blank entries. Every blank belongs to a customer
with `tenure = 0` — signed up, never billed. That is not missing data, it is a real
zero, and imputing the column mean would hand a brand-new customer roughly $2,280 of
fabricated billing history and make them look like a loyal long-timer to the model.

## 2. The target, and why accuracy is the wrong metric

In [3]:
rate = df.churn_flag.mean()
print(f'churn rate      : {rate:.2%}')
print(f'churned         : {df.churn_flag.sum():,} of {len(df):,}')
print(f'imbalance ratio : {(1-rate)/rate:.2f} : 1')
print(f'"nobody churns" accuracy baseline: {1-rate:.2%}')

churn rate      : 26.54%
churned         : 1,869 of 7,043
imbalance ratio : 2.77 : 1
"nobody churns" accuracy baseline: 73.46%


A model that predicts *nobody churns* scores **73.5% accuracy** and is worth nothing.
That single number is why this project is graded on recall and ROC-AUC.

## 3. Revenue at risk — the headline the Finance team asked for

In [4]:
churned = df[df.churn_flag == 1]
mrr = churned.monthly_charges.sum()
print(f'monthly recurring revenue lost : ${mrr:,.0f}')
print(f'annualised                     : ${mrr*12:,.0f}')
print(f'avg bill, churner vs retained  : ${churned.monthly_charges.mean():.2f} '
      f'vs ${df[df.churn_flag==0].monthly_charges.mean():.2f}')
print(f'avg tenure, churner vs retained: {churned.tenure.mean():.1f} '
      f'vs {df[df.churn_flag==0].tenure.mean():.1f} months')

monthly recurring revenue lost : $139,131
annualised                     : $1,669,570
avg bill, churner vs retained  : $74.44 vs $61.27
avg tenure, churner vs retained: 18.0 vs 37.6 months


Churners pay **more** and stay **half as long**. Churn is not concentrated in cheap
accounts — it is eating the premium book.

## 4. Bivariate — churn rate against every categorical driver

In [5]:
for col in ['contract', 'internet_service', 'payment_method', 'tech_support']:
    t = (df.groupby(col)['churn_flag']
           .agg(customers='size', churn_rate='mean')
           .sort_values('churn_rate', ascending=False))
    t['churn_rate'] = (t.churn_rate*100).round(2)
    print(f'\n--- {col} ---')
    print(t.to_string())


--- contract ---
                customers  churn_rate
contract                             
Month-to-month       3875       42.71
One year             1473       11.27
Two year             1695        2.83

--- internet_service ---
                  customers  churn_rate
internet_service                       
Fiber optic            3096       41.89
DSL                    2421       18.96
No                     1526        7.40

--- payment_method ---
                           customers  churn_rate
payment_method                                  
Electronic check                2365       45.29
Mailed check                    1612       19.11
Bank transfer (automatic)       1544       16.71
Credit card (automatic)         1522       15.24

--- tech_support ---
                     customers  churn_rate
tech_support                              
No                        3473       41.64
Yes                       2044       15.17
No internet service       1526        7.40


## 5. Correlation and multicollinearity

In [6]:
from features import engineer_features
feats = engineer_features(df)
num = feats.select_dtypes(include=[np.number]).copy()
num['churn_flag'] = df.churn_flag.values
corr = num.corr(numeric_only=True)
corr['churn_flag'].drop('churn_flag').sort_values().to_frame('r with churn')

,r with churn
contract_ord,-0.396713
tenure,-0.352229
tenure_group_ord,-0.345410
total_charges,-0.198324
protection_services,-0.173061
total_services,-0.087698
charge_trend_delta,0.002159
senior_citizen,0.150889
avg_monthly_spend_ratio,0.192531
monthly_charges,0.193356


In [7]:
cols = [c for c in corr.columns if c != 'churn_flag']
pairs = [(a, b, round(corr.loc[a, b], 3))
         for i, a in enumerate(cols) for b in cols[i+1:] if abs(corr.loc[a, b]) > 0.8]
print('|r| > 0.8 between predictors:')
for a, b, r in pairs:
    print(f'  {a:<26} <-> {b:<26} r={r}')

|r| > 0.8 between predictors:
  tenure                     <-> total_charges              r=0.826
  tenure                     <-> tenure_group_ord           r=0.961
  monthly_charges            <-> avg_monthly_spend_ratio    r=0.996
  total_services             <-> protection_services        r=0.913


`tenure` and `total_charges` move together (r=0.83) because total billing is
roughly price x months — unavoidable and expected. This matters for the linear
baseline (inflated coefficient variance, which L2 regularisation absorbs) and is
irrelevant to the tree models, which is part of why a tree ensemble was chosen.

## 6. Segment analysis — where the loss actually concentrates

In [8]:
d = df.copy()
d['tenure_group'] = pd.cut(d.tenure, [-0.1, 12, 24, 48, np.inf],
                           labels=['0-12', '13-24', '25-48', '49+']).astype(str)
pivot = (d.pivot_table(index='contract', columns='tenure_group',
                       values='churn_flag', aggfunc='mean')*100).round(1)
print(pivot[['0-12', '13-24', '25-48', '49+']].to_string())

tenure_group    0-12  13-24  25-48   49+
contract                                
Month-to-month  51.4   37.7   32.9  26.0
One year        10.5    8.1   10.6  12.9
Two year         0.0    0.0    2.2   3.3

In [9]:
seg = d[(d.tenure < 6) & (d.contract == 'Month-to-month') & (d.tech_support == 'No')]
print(f'new + month-to-month + no tech support')
print(f'  customers   : {len(seg):,}')
print(f'  churn rate  : {seg.churn_flag.mean():.1%}')
print(f'  MRR at risk : ${seg[seg.churn_flag==1].monthly_charges.sum():,.0f}/month')

new + month-to-month + no tech support
  customers   : 904
  churn rate  : 66.7%
  MRR at risk : $40,837/month


## Insights summary — as it would be emailed to the VP of Retention

1. **Churn is 26.5%**, worth **$139,131/month** — about **$1.67M a year**.
2. **Contract type is the lever.** Month-to-month churns at 42.7%, two-year at 2.8%.
3. **The first year is where they leave.** 47.4% in months 0–12, 9.5% after four years.
4. **The worst pocket:** new + month-to-month + no tech support — **66.7% churn**
   across 904 customers.
5. **Fiber is a problem product** — 41.9% churn on a $91.50 average bill vs 19.0% on DSL.
6. **Manual payers leave.** Electronic check churns at 45.3% vs 15.2% on autopay.
7. **Depth protects.** Six add-ons: 5.3% churn. One add-on: 45.8%.